In [ ]:
!pip install transformers torch datasets

# Preview Dataset

In [6]:
from datasets import load_dataset_builder

# Allow us to access a dataset's info without download it
ds_builder = load_dataset_builder("knkarthick/samsum")
ds_builder.info

DatasetInfo(description='', citation='', homepage='', license='', features={'id': Value('string'), 'dialogue': Value('string'), 'summary': Value('string')}, post_processed=None, supervised_keys=None, builder_name='csv', dataset_name='samsum', config_name='default', version=0.0.0, splits={'train': SplitInfo(name='train', num_bytes=9347792, num_examples=14731, shard_lengths=None, dataset_name='samsum'), 'validation': SplitInfo(name='validation', num_bytes=509370, num_examples=818, shard_lengths=None, dataset_name='samsum'), 'test': SplitInfo(name='test', num_bytes=526994, num_examples=819, shard_lengths=None, dataset_name='samsum')}, download_checksums={'hf://datasets/knkarthick/samsum@6b929ff10edec703164e3ddb2e94aae058c9ab5f/train.csv': {'num_bytes': 9255369, 'checksum': None}, 'hf://datasets/knkarthick/samsum@6b929ff10edec703164e3ddb2e94aae058c9ab5f/validation.csv': {'num_bytes': 504263, 'checksum': None}, 'hf://datasets/knkarthick/samsum@6b929ff10edec703164e3ddb2e94aae058c9ab5f/test.c

In [ ]:
!pip install py7zr

In [8]:
from datasets import load_dataset
# The original samsum is not availble now, use knkarthick/samsum instead
dataset = load_dataset("knkarthick/samsum")
dataset

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

# Data Processing Models

In [9]:
from datasets import load_dataset

# Here we actually download the dataset
# Returns a DatasetDict object
# This dataset only have train set, others will include validation and test as well
dataset = load_dataset("fka/awesome-chatgpt-prompts")
dataset

README.md: 0.00B [00:00, ?B/s]

prompts.csv: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['act', 'prompt', 'for_devs', 'type', 'contributor'],
        num_rows: 1468
    })
})

In [10]:
# Print an example
dataset["train"][0]

{'act': 'Ethereum Developer',
 'prompt': 'Imagine you are an experienced Ethereum developer tasked with creating a smart contract for a blockchain messenger. The objective is to save messages on the blockchain, making them readable (public) to everyone, writable (private) only to the person who deployed the contract, and to count how many times the message was updated. Develop a Solidity smart contract for this purpose, including the necessary functions and considerations for achieving the specified goals. Please provide the code and any relevant explanations to ensure a clear understanding of the implementation.',
 'for_devs': True,
 'type': 'TEXT',
 'contributor': 'ameya-2003'}

In [11]:
# Shuffle & Sample
# Get a dataset containing 100 rows, seed is identified to make sure the reproducibility
dataset = dataset['train'].shuffle(seed=37).select(range(100))
dataset

Dataset({
    features: ['act', 'prompt', 'for_devs', 'type', 'contributor'],
    num_rows: 100
})

In [12]:
# Create Test Dataset
# Here, our dataset is splited into train and test sub-sets
dataset = dataset.train_test_split(train_size=0.8, seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['act', 'prompt', 'for_devs', 'type', 'contributor'],
        num_rows: 80
    })
    test: Dataset({
        features: ['act', 'prompt', 'for_devs', 'type', 'contributor'],
        num_rows: 20
    })
})

# Creating Your Own Dataset

In [13]:
!wget https://kdd.ics.uci.edu/databases/reuters21578/reuters21578.tar.gz

--2026-03-16 11:33:45--  https://kdd.ics.uci.edu/databases/reuters21578/reuters21578.tar.gz
Resolving kdd.ics.uci.edu (kdd.ics.uci.edu)... 128.195.1.94
Connecting to kdd.ics.uci.edu (kdd.ics.uci.edu)|128.195.1.94|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8150596 (7.8M) [application/x-gzip]
Saving to: ‘reuters21578.tar.gz’

reuters21578.tar.gz 100%[===================>]   7.77M  14.6MB/s    in 0.5s    

2026-03-16 11:33:46 (14.6 MB/s) - ‘reuters21578.tar.gz’ saved [8150596/8150596]



In [14]:
# the sgm files contains the articles (text messages)
!tar -xzvf reuters21578.tar.gz

README.txt
all-exchanges-strings.lc.txt
all-orgs-strings.lc.txt
all-people-strings.lc.txt
all-places-strings.lc.txt
all-topics-strings.lc.txt
cat-descriptions_120396.txt
feldman-cia-worldfactbook-data.txt
lewis.dtd
reut2-000.sgm
reut2-001.sgm
reut2-002.sgm
reut2-003.sgm
reut2-004.sgm
reut2-005.sgm
reut2-006.sgm
reut2-007.sgm
reut2-008.sgm
reut2-009.sgm
reut2-010.sgm
reut2-011.sgm
reut2-012.sgm
reut2-013.sgm
reut2-014.sgm
reut2-015.sgm
reut2-016.sgm
reut2-017.sgm
reut2-018.sgm
reut2-019.sgm
reut2-020.sgm
reut2-021.sgm


In [15]:
from bs4 import BeautifulSoup

reuters_articles = []
for i in range(22):
  if i < 10:
    i = f"0{i}"

  # Load file data
  with open(f"/content/reut2-0{i}.sgm", "r", encoding="latin-1") as file:
    soup = BeautifulSoup(file, "html.parser")

  # Extract articles' titles and bodies
  articles = []
  for reuters in soup.find_all("reuters"):
    title = reuters.title.string if reuters.title else ""
    body = reuters.body.string if reuters.body else ""
    articles.append({
        "title": title,
        "body": body
    })

  reuters_articles.extend(articles)


In [16]:
# Print the first few articles for inspection
for i, article in enumerate(reuters_articles[:5]):
  print(f"Article no. {i}")
  print(article)
  print("-"*100)

Article no. 0
{'title': 'BAHIA COCOA REVIEW', 'body': 'Showers continued throughout the week in\nthe Bahia cocoa zone, alleviating the drought since early\nJanuary and improving prospects for the coming temporao,\nalthough normal humidity levels have not been restored,\nComissaria Smith said in its weekly review.\n    The dry period means the temporao will be late this year.\n    Arrivals for the week ended February 22 were 155,221 bags\nof 60 kilos making a cumulative total for the season of 5.93\nmln against 5.81 at the same stage last year. Again it seems\nthat cocoa delivered earlier on consignment was included in the\narrivals figures.\n    Comissaria Smith said there is still some doubt as to how\nmuch old crop cocoa is still available as harvesting has\npractically come to an end. With total Bahia crop estimates\naround 6.4 mln bags and sales standing at almost 6.2 mln there\nare a few hundred thousand bags still in the hands of farmers,\nmiddlemen, exporters and processors.\n  

In [17]:
# length of the Reuters Articles
len(reuters_articles)

21578

In [18]:
import json

TRAIN_PCT, VALID_PCT = 0.8, 0.1
TRAIN_NUM, VALID_NUM = int(len(reuters_articles)*TRAIN_PCT), int(len(reuters_articles)*VALID_PCT)

# Split the data
train_articles = reuters_articles[:TRAIN_NUM]
valid_articles = reuters_articles[TRAIN_NUM:TRAIN_NUM+VALID_NUM]
test_articles = reuters_articles[TRAIN_NUM+VALID_NUM:]

print(len(reuters_articles), len(train_articles), len(valid_articles), len(test_articles))

21578 17262 2157 2159


In [21]:
# Function to save articles as JSONL

def save_as_jsonl(data, filename):
  with open(filename, "w") as f:
    for article in data:
      f.write(json.dumps(article) + "\n")

# Save them into temporary JSONL files
save_as_jsonl(train_articles, "train.jsonl")
save_as_jsonl(valid_articles, "valid.jsonl")
save_as_jsonl(test_articles, "test.jsonl")

In [23]:
# Load them as variables
data_files = {"train": "train.jsonl",
              "validation":"valid.jsonl",
              "test": "test.jsonl"}

dataset = load_dataset("json", data_files=data_files)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [24]:
dataset

DatasetDict({
    train: Dataset({
        features: ['title', 'body'],
        num_rows: 17262
    })
    validation: Dataset({
        features: ['title', 'body'],
        num_rows: 2157
    })
    test: Dataset({
        features: ['title', 'body'],
        num_rows: 2159
    })
})

In [25]:
dataset["train"][0]

{'title': 'BAHIA COCOA REVIEW',
 'body': 'Showers continued throughout the week in\nthe Bahia cocoa zone, alleviating the drought since early\nJanuary and improving prospects for the coming temporao,\nalthough normal humidity levels have not been restored,\nComissaria Smith said in its weekly review.\n    The dry period means the temporao will be late this year.\n    Arrivals for the week ended February 22 were 155,221 bags\nof 60 kilos making a cumulative total for the season of 5.93\nmln against 5.81 at the same stage last year. Again it seems\nthat cocoa delivered earlier on consignment was included in the\narrivals figures.\n    Comissaria Smith said there is still some doubt as to how\nmuch old crop cocoa is still available as harvesting has\npractically come to an end. With total Bahia crop estimates\naround 6.4 mln bags and sales standing at almost 6.2 mln there\nare a few hundred thousand bags still in the hands of farmers,\nmiddlemen, exporters and processors.\n    There are d

# Upload Dataset to Hub

In [27]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
dataset.push_to_hub("reuters_articles")